# Notebook 3 — Simple ETL Pipeline

**Use Case 2 goal:** demonstrate the ETL pattern end to end — **Extract → Transform → Aggregate → Load**.

## What learners should understand
- how the cleaned file becomes an ETL input
- how aggregation turns row-level transactions into reporting-ready outputs
- how the same business logic can later be recreated in AWS Glue with PySpark


## Dependencies, AWS setup, and files used

### Python packages
- **boto3** — AWS-friendly access to the cleaned file in S3 and the final ETL outputs written back to S3
- **pandas** — easiest way to explain ETL aggregation logic cell by cell
- **io** — bridges S3 object content and pandas DataFrames
- **pathlib** — fallback for offline rehearsal


In [7]:
from pathlib import Path
from io import BytesIO, StringIO
import boto3
import pandas as pd

AWS_REGION = boto3.session.Session().region_name or 'ap-south-1'

# ✅ Direct S3 paths (no env vars, no fallback confusion)
INPUT_S3_URI = "s3://usecase-etl-1/processed/retail_cleaned.csv"
OUT_DAILY_S3_URI = "s3://usecase-etl-2/output/daily_country_revenue.csv"
OUT_MONTHLY_S3_URI = "s3://usecase-etl-2/output/monthly_category_revenue.csv"

# Local fallback (only if needed)
LOCAL_INPUT_PATH = Path('./retail_cleaned.csv')
LOCAL_DAILY_PATH = Path('./daily_country_revenue.csv')
LOCAL_MONTHLY_PATH = Path('./monthly_category_revenue.csv')

s3_client = boto3.client('s3', region_name=AWS_REGION)


def parse_s3_uri(uri: str):
    bucket, key = uri.replace('s3://', '', 1).split('/', 1)
    return bucket, key


def read_csv_aws_first(s3_uri: str, local_path: Path) -> pd.DataFrame:
    try:
        bucket, key = parse_s3_uri(s3_uri)
        obj = s3_client.get_object(Bucket=bucket, Key=key)
        print("✅ Reading from S3:", s3_uri)
        return pd.read_csv(BytesIO(obj['Body'].read()))
    except Exception as e:
        print(f"⚠️ S3 failed: {e}")
        print("📂 Falling back to local:", local_path)
        return pd.read_csv(local_path)


def write_csv_aws_first(df: pd.DataFrame, s3_uri: str, local_path: Path) -> None:
    csv_buffer = StringIO()
    df.to_csv(csv_buffer, index=False)

    try:
        bucket, key = parse_s3_uri(s3_uri)
        s3_client.put_object(Bucket=bucket, Key=key, Body=csv_buffer.getvalue().encode('utf-8'))
        print("✅ Written to S3:", s3_uri)
    except Exception as e:
        print(f"⚠️ S3 write failed: {e}")
        print("📂 Writing locally:", local_path)
        local_path.parent.mkdir(parents=True, exist_ok=True)
        local_path.write_text(csv_buffer.getvalue(), encoding='utf-8')


print('ETL input:', INPUT_S3_URI)
print('Daily output:', OUT_DAILY_S3_URI)
print('Monthly output:', OUT_MONTHLY_S3_URI)

ETL input: s3://usecase-etl-1/processed/retail_cleaned.csv
Daily output: s3://usecase-etl-2/output/daily_country_revenue.csv
Monthly output: s3://usecase-etl-2/output/monthly_category_revenue.csv


## Step 1 — Extract the cleaned data

### Why this step is performed
In ETL terms, this is the **extract** phase: we load the cleaned source that is ready for downstream calculations and aggregation.


In [8]:
df = read_csv_aws_first(INPUT_S3_URI, LOCAL_INPUT_PATH)
print('Clean input shape:', df.shape)
display(df.head())


✅ Reading from S3: s3://usecase-etl-1/processed/retail_cleaned.csv
Clean input shape: (489, 14)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,InvoiceDateParsed,TransactionDate,Year,Month,Revenue,IsReturn
0,536365,71053,WHITE METAL LANTERN,6,2011-02-01 11:08:00,5.49,17889.0,Belgium,2011-02-01 11:08:00,2011-02-01,2011,2011-02,32.94,False
1,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,2,2011-01-28 11:32:00,4.22,16943.0,Germany,2011-01-28 11:32:00,2011-01-28,2011,2011-01,8.44,False
2,536366,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2011-02-05 08:48:00,5.80,18065.0,Netherlands,2011-02-05 08:48:00,2011-02-05,2011,2011-02,34.80,False
3,536366,22752,SET 7 BABUSHKA NESTING BOXES,4,2011-01-13 13:54:00,7.55,14512.0,United Kingdom,2011-01-13 13:54:00,2011-01-13,2011,2011-01,30.20,False
4,536366,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2011-03-22 17:56:00,4.03,17075.0,Germany,2011-03-22 17:56:00,2011-03-22,2011,2011-03,24.18,False


## Step 2 — Transform for reporting

### Why this step is performed
Create fields that make aggregation easier.


In [9]:
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'])
df['Category'] = df['Description'].fillna('UNKNOWN_ITEM').str.split().str[0]
df['Revenue'] = df['Quantity'] * df['UnitPrice']

display(df[['TransactionDate', 'Country', 'Category', 'Revenue']].head())


,TransactionDate,Country,Category,Revenue
0,2011-02-01,Belgium,WHITE,32.94
1,2011-01-28,Germany,GLASS,8.44
2,2011-02-05,Netherlands,GLASS,34.80
3,2011-01-13,United Kingdom,SET,30.20
4,2011-03-22,Germany,GLASS,24.18


## Step 3 — Aggregate into reporting outputs

### Why this step is performed
This is where row-level data becomes business-ready output.


In [10]:
daily_country_revenue = (
    df.groupby([df['TransactionDate'].dt.date.astype(str), 'Country'], as_index=False)['Revenue']
      .sum()
      .rename(columns={'TransactionDate': 'Date'})
)

monthly_category_revenue = (
    df.groupby(['Month', 'Category'], as_index=False)['Revenue']
      .sum()
)

display(daily_country_revenue.head())
display(monthly_category_revenue.head())


,Country,Revenue
0,France,7.74
1,United Kingdom,63.96
2,Belgium,20.75
3,Germany,50.78
4,Belgium,11.62


,Month,Category,Revenue
0,2011-01,ASSORTED,333.42
1,2011-01,CREAM,314.82
2,2011-01,GLASS,432.14
3,2011-01,HAND,1161.19
4,2011-01,KNITTED,558.09


## Step 4 — Load the ETL outputs back to S3

### Why this step is performed
The daily and monthly views are written back to S3 so they can be verified in the AWS console.


In [11]:
write_csv_aws_first(daily_country_revenue, OUT_DAILY_S3_URI, LOCAL_DAILY_PATH)
write_csv_aws_first(monthly_category_revenue, OUT_MONTHLY_S3_URI, LOCAL_MONTHLY_PATH)

print('Daily revenue output saved to:')
print(OUT_DAILY_S3_URI if not USE_LOCAL_FALLBACK else LOCAL_DAILY_PATH.resolve())
print('Monthly revenue output saved to:')
print(OUT_MONTHLY_S3_URI if not USE_LOCAL_FALLBACK else LOCAL_MONTHLY_PATH.resolve())


✅ Written to S3: s3://usecase-etl-2/output/daily_country_revenue.csv
✅ Written to S3: s3://usecase-etl-2/output/monthly_category_revenue.csv
Daily revenue output saved to:
/mnt/custom-file-systems/s3/shared/daily_country_revenue.csv
Monthly revenue output saved to:
/mnt/custom-file-systems/s3/shared/monthly_category_revenue.csv
